Tarea: Proponer un metodo para distribuir k centroides en la
Imagen (El Astronomo, 0000405625_OG.JPG) de tal forma que la imagen obtenida a
partir de los Centroides sea lo mas parecida A la imagen original
¿Que funcion de distancia utiizaras para comparar la imagen original
Con la obtenida por el metodo (teselacion de Voronoi)?

use image vector quantization techniques and build it from scratch (explain and imeplemnet: how are the representative points selected, search for representative points etc.)

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import time
from scipy.spatial import Voronoi, voronoi_plot_2d

# Function to load the image
def load_image(image_path):
    img = Image.open(image_path)
    img_array = np.array(img)
    return img_array

# Implement K-means++ initialization for better centroid placement
def kmeans_plus_plus_init(data, k):
    n_samples, n_features = data.shape
    centroids = np.zeros((k, n_features))
    
    # Choose first centroid randomly
    centroids[0] = data[np.random.randint(0, n_samples)]
    
    # Choose remaining centroids with probability proportional to distance squared
    for i in range(1, k):
        # Calculate squared distances to nearest centroids
        min_distances = np.min([np.sum((data - centroids[j])**2, axis=1) 
                              for j in range(i)], axis=0)
        
        # Select next centroid with probability proportional to distance squared
        probs = min_distances / np.sum(min_distances)
        cumprobs = np.cumsum(probs)
        r = np.random.random()
        ind = np.searchsorted(cumprobs, r)
        centroids[i] = data[ind]
    
    return centroids

# Implement K-means clustering from scratch
def kmeans(data, k, max_iters=100, tol=1e-4):
    n_samples = data.shape[0]
    
    # Initialize centroids using K-means++
    centroids = kmeans_plus_plus_init(data, k)
    
    # Initialize labels and distances
    labels = np.zeros(n_samples, dtype=int)
    
    for iter_num in range(max_iters):
        old_centroids = centroids.copy()
        
        # Assign each point to the nearest centroid
        for i in range(n_samples):
            distances = np.sqrt(np.sum((centroids - data[i])**2, axis=1))
            labels[i] = np.argmin(distances)
        
        # Update centroids
        for j in range(k):
            cluster_points = data[labels == j]
            if len(cluster_points) > 0:
                centroids[j] = np.mean(cluster_points, axis=0)
        
        # Check for convergence
        if np.sum((centroids - old_centroids)**2) < tol:
            print(f"K-means converged after {iter_num+1} iterations")
            break
    
    return centroids, labels

# Create a Voronoi image from centroids and labels
def create_voronoi_image(img_shape, centroids, labels):
    height, width, channels = img_shape
    voronoi_img = np.zeros(img_shape, dtype=np.uint8)
    
    # Map each pixel to its centroid color
    pixel_indices = np.arange(height * width)
    row_indices = pixel_indices // width
    col_indices = pixel_indices % width
    
    for i, label in enumerate(labels):
        row, col = row_indices[i], col_indices[i]
        voronoi_img[row, col] = centroids[label]
    
    return voronoi_img

# Compute Mean Squared Error between original and quantized images
def compute_mse(original, quantized):
    return np.mean((original.astype(float) - quantized.astype(float)) ** 2)

# Compute Peak Signal-to-Noise Ratio
def compute_psnr(original, quantized):
    mse = compute_mse(original, quantized)
    if mse == 0:
        return float('inf')
    return 20 * np.log10(255.0 / np.sqrt(mse))

# Main function to perform image vector quantization
def vector_quantize_image(image_path, k):
    print(f"Performing vector quantization with k={k} centroids...")
    
    # Load the image
    img_array = load_image(image_path)
    
    # Reshape the image to a list of pixels
    pixels = img_array.reshape(-1, 3)  # 3 channels for RGB
    
    # Perform K-means clustering
    start_time = time.time()
    centroids, labels = kmeans(pixels, k)
    processing_time = time.time() - start_time
    
    # Create the Voronoi image
    voronoi_img = create_voronoi_image(img_array.shape, centroids, labels)
    
    # Compute quality metrics
    mse = compute_mse(img_array, voronoi_img)
    psnr = compute_psnr(img_array, voronoi_img)
    
    print(f"K-means with k={k} completed in {processing_time:.2f} seconds")
    print(f"MSE: {mse:.2f}, PSNR: {psnr:.2f} dB")
    
    return img_array, voronoi_img, centroids, labels, mse, psnr

# Visualize the results
def visualize_results(original, quantized, k, psnr):
    plt.figure(figsize=(15, 7))
    
    plt.subplot(1, 2, 1)
    plt.imshow(original)
    plt.title("Original Image")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(quantized)
    plt.title(f"Vector Quantized Image (k={k}, PSNR={psnr:.2f} dB)")
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize the color space of centroids
def visualize_color_centroids(centroids):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot the centroids in RGB space
    ax.scatter(centroids[:, 0], centroids[:, 1], centroids[:, 2], 
               c=centroids/255, s=100)
    
    ax.set_xlabel('Red')
    ax.set_ylabel('Green')
    ax.set_zlabel('Blue')
    ax.set_title('Color Centroids in RGB Space')
    plt.show()

# Example usage
image_path = "0000405625_OG.JPG"
k = 32  # Number of centroids

# Run vector quantization
original, quantized, centroids, labels, mse, psnr = vector_quantize_image(image_path, k)

# Visualize results
visualize_results(original, quantized, k, psnr)
visualize_color_centroids(centroids)

Performing vector quantization with k=32 centroids...


FileNotFoundError: [Errno 2] No such file or directory: '0000405625_OG.JPG'